# A. Persiapan Data & Train-Test Split

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Library Pemodelan
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report, roc_curve)

# Algoritma
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Penanganan Imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import os, warnings
warnings.filterwarnings("ignore")

## 1. Load Dataset: Membaca dataset final yang sudah di‑engineering ***hr_final_dataset.csv***

In [2]:
# ============================================================
# 1. LOAD DATA: Membaca dataset final yang sudah di‑engineering
#    Sumber: hr_final_dataset.csv  yang ada pada Google Drive.
# ============================================================

file_id_gd = '1XBCIZJHpn466q2B0tdyebFwNYsYEZ7ij'
file_name_gd = 'hr_final_dataset.csv'
# https://drive.google.com/file/d/1XBCIZJHpn466q2B0tdyebFwNYsYEZ7ij/view?usp=sharing
!gdown --id {file_id_gd} --output {file_name_gd}
df = pd.read_csv(file_name_gd)

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1XBCIZJHpn466q2B0tdyebFwNYsYEZ7ij
To: /content/hr_final_dataset.csv
100% 3.32M/3.32M [00:00<00:00, 44.9MB/s]


In [3]:
# Inspect columns
cols = df.columns.tolist()
shape = df.shape
shape_str = f"Baris: {shape[0]}, Kolom: {shape[1]}"
shape_str

'Baris: 4410, Kolom: 67'

## 2. Preprocessing Data (Pemisahan Fitur dan Target)

In [4]:
# Cek & Drop EmployeeID jika ada (karena tidak prediktif)
if 'EmployeeID' in df.columns:
    df = df.drop('EmployeeID', axis=1)

# Pisahkan Fitur (X) dan Target (y)
X = df.drop('AttritionFlag', axis=1)
y = df['AttritionFlag']

## 3. Train-Test Split (80:20) dengan Stratifikasi

In [5]:
# 2. Train-Test Split (80:20) dengan Stratifikasi
# Stratify=y memastikan proporsi attrition di train dan test sama
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Shape Train: {X_train.shape}, Shape Test: {X_test.shape}")
print(f"Distribusi Target Train:\n{y_train.value_counts(normalize=True)}")

Shape Train: (3528, 65), Shape Test: (882, 65)
Distribusi Target Train:
AttritionFlag
0    0.838719
1    0.161281
Name: proportion, dtype: float64


## 4. Model Benchmarking

In [6]:
# Model Benchmarking (Multi-Metric Cross-Validation)
# ==============================================================================

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score
)

# --- Definisi Custom Metrics ---

def precision_at_top_k(y_true, y_probs, k=0.1):
    """Menghitung Precision pada Top K% probabilitas tertinggi"""
    n = len(y_probs)
    k_count = int(n * k)
    # Ambil index dari probabilitas tertinggi
    top_k_indices = np.argsort(y_probs)[-k_count:]
    # Ambil label aktual pada index tersebut
    y_true_top_k = np.array(y_true)[top_k_indices]
    return np.sum(y_true_top_k) / k_count

# Buat Scorer untuk Cross-Validation
# Note: PR-AUC menggunakan average_precision_score
prec_at_10_scorer = make_scorer(precision_at_top_k, response_method='predict_proba', k=0.1)
pr_auc_scorer = make_scorer(average_precision_score, response_method='predict_proba')
f2_scorer = make_scorer(fbeta_score, beta=2)

scoring_metrics = {
    'Accuracy': 'accuracy',
    'Recall': 'recall',
    'Precision': 'precision',
    'Precision@Top10%': prec_at_10_scorer,
    'F1': 'f1',
    'F2': f2_scorer,
    'ROC_AUC': 'roc_auc',
    'PR_AUC': pr_auc_scorer
}

In [7]:
# Inisialisasi Model Dasar
models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "KNN": KNeighborsClassifier()
}

In [8]:
# List untuk menampung hasil rata-rata
benchmark_results = []

print("Mulai Benchmarking Model dengan 5-Fold CV (Stratified)...\n")

for name, model in models.items():
    # Pipeline: Scaling -> SMOTE -> Model
    # Penting: SMOTE hanya dilakukan pada data Train di setiap fold (mencegah data leakage)
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    # Lakukan Cross-Validate
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,                 # 5-Fold Stratified secara default untuk klasifikasi
        scoring=scoring_metrics,
        n_jobs=-1             # Gunakan semua core processor
    )

    # Ambil rata-rata dari setiap metrik
    result_entry = {
        'Model': name,
        'Accuracy': cv_results['test_Accuracy'].mean(),
        'Precision': cv_results['test_Precision'].mean(),
        'Precision@Top10%': cv_results['test_Precision@Top10%'].mean(),
        'Recall': cv_results['test_Recall'].mean(),
        'F1-Score': cv_results['test_F1'].mean(),
        'F2-Score': cv_results['test_F2'].mean(),
        'ROC-AUC': cv_results['test_ROC_AUC'].mean(),
        'PR-AUC': cv_results['test_PR_AUC'].mean()
    }

    benchmark_results.append(result_entry)
    print(f"--> {name} selesai dievaluasi.")

Mulai Benchmarking Model dengan 5-Fold CV (Stratified)...

--> Logistic Regression selesai dievaluasi.
--> Decision Tree selesai dievaluasi.
--> Random Forest selesai dievaluasi.
--> XGBoost selesai dievaluasi.
--> KNN selesai dievaluasi.


In [10]:
# Menampilkan Hasil dalam DataFrame
benchmark_df = pd.DataFrame(benchmark_results)

# Urutkan berdasarkan ROC-AUC (atau F2-Score sesuai prioritas bisnis)
benchmark_df = benchmark_df.sort_values(by='ROC-AUC', ascending=False)

print("\n=== HASIL BENCHMARKING MODEL (Rata-rata 5-Fold CV) ===")
# Format tampilan agar angka desimal rapi
pd.options.display.float_format = '{:,.2%}'.format
benchmark_df


=== HASIL BENCHMARKING MODEL (Rata-rata 5-Fold CV) ===


,Model,Accuracy,Precision,Precision@Top10%,Recall,F1-Score,F2-Score,ROC-AUC,PR-AUC
2,Random Forest,97.99%,96.83%,100.00%,90.51%,93.53%,91.69%,99.39%,97.81%
3,XGBoost,97.53%,94.62%,99.71%,89.81%,92.14%,90.73%,98.11%,96.39%
4,KNN,79.62%,43.88%,88.29%,94.03%,59.82%,76.51%,94.05%,79.45%
1,Decision Tree,96.15%,87.99%,88.00%,88.24%,88.06%,88.16%,92.95%,79.55%
0,Logistic Regression,76.50%,37.84%,61.71%,70.29%,49.17%,59.96%,81.50%,52.19%


In [ ]:
# Hyperparameter Tuning (All Models) dengan Multi-Metric Report
# ==============================================================================

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, fbeta_score

param_grids = {
    "Logistic Regression": {'model__C': [0.01, 0.1, 1, 10, 100], 'model__penalty': ['l1', 'l2']},
    "Decision Tree": {'model__max_depth': [None, 5, 10, 20], 'model__min_samples_leaf': [1, 2, 4]},
    "Random Forest": {'model__n_estimators': [100, 200, 300], 'model__max_depth': [10, 20, None], 'model__min_samples_leaf': [1, 2, 4]},
    "XGBoost": {'model__n_estimators': [100, 200, 300], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__max_depth': [3, 5, 7], 'model__subsample': [0.7, 0.8, 1.0]},
    "KNN": {'model__n_neighbors': [3, 5, 7, 9], 'model__weights': ['uniform', 'distance']}
}

tuning_results = []
best_estimators = {}

print("Mulai Hyperparameter Tuning (Optimasi: ROC-AUC, Pantau: Semua Metrik)")

for name, model in models.items():
    print(f"Sedang tuning: {name}...")

    # Pipeline
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    # Setup Search
    # refit='roc_auc' artinya model final akan dipilih berdasarkan skor ROC-AUC terbaik
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grids[name],
        n_iter=15,
        cv=5,
        scoring=scoring_metrics, # Pantau semua metrik ini
        refit='ROC_AUC',         # Optimalkan berdasarkan ini
        n_jobs=-1,
        random_state=42,
        verbose=0
    )

    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_

    # Mengambil hasil untuk kombinasi parameter terbaik
    # Kita harus mencari index dari skor ROC-AUC terbaik
    best_idx = search.best_index_

    # Simpan hasil metrik validasi dari model terbaik tersebut
    tuning_results.append({
        'Model': name,
        'Accuracy': search.cv_results_['mean_test_Accuracy'][best_idx],
        'Precision': search.cv_results_['mean_test_Precision'][best_idx],
        'Precision@Top10%': search.cv_results_['mean_test_Precision@Top10%'][best_idx],
        'Recall': search.cv_results_['mean_test_Recall'][best_idx],
        'F1-Score': search.cv_results_['mean_test_F1'][best_idx],
        'F2-Score': search.cv_results_['mean_test_F2'][best_idx],
        'ROC-AUC': search.cv_results_['mean_test_ROC_AUC'][best_idx],
        'PR-AUC': search.cv_results_['mean_test_PR_AUC'][best_idx],
        'Best Params': search.best_params_
    })

    print(f"  --> {name} Selesai. Best ROC-AUC: {search.best_score_:.4f}")

In [ ]:
# Evaluasi: Leaderboard & Final Test
# ==============================================================================

# Membuat Leaderboard DataFrame
tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values(by='ROC-AUC', ascending=False)

print("\n=== LEADERBOARD MODEL SETELAH TUNING (Validation Metrics) ===")
pd.options.display.float_format = '{:,.2%}'.format
# Tampilkan metrik utama saja agar rapi, params bisa dilihat terpisah jika perlu
display_cols = ['Model', 'Accuracy', 'Precision', 'Precision@Top10%', 'Recall', 'F1-Score', 'F2-Score', 'ROC-AUC', 'PR-AUC']
print(tuning_df[display_cols])

# Memilih Model Juara (Baris pertama dari leaderboard)
champion_name = tuning_df.iloc[0]['Model']
champion_model = best_estimators[champion_name]

print(f"\nModel Terpilih: {champion_name}")
print("Melakukan Evaluasi Final pada Test Set (Data Asing)...")

# Prediksi Final
y_pred_test = champion_model.predict(X_test)
y_pred_proba_test = champion_model.predict_proba(X_test)[:, 1]

# Tampilkan Hasil Final
print("\n=== FINAL PERFORMANCE (TEST SET) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_test):.2%}")
print(f"Precision : {precision_score(y_test, y_pred_test):.2%}")
print(f"Precision@10% : {precision_at_top_k(y_test, y_pred_proba_test, k=0.1):.2%}")
print(f"Recall    : {recall_score(y_test, y_pred_test):.2%}")
print(f"F1-Score  : {f1_score(y_test, y_pred_test):.2%}")
print(f"F2-Score  : {fbeta_score(y_test, y_pred_test, beta=2):.2%}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_pred_proba_test):.2%}")
print(f"PR-AUC    : {average_precision_score(y_test, y_pred_proba_test):.2%}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_test))

## 5. Visualisasi

In [ ]:
# ==============================================================================
# 8. Visualisasi Komprehensif (Confusion Matrix, Learning Curve, Feature Imp.)
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, auc

# Setup Canvas (Dashboard Layout: 2 Baris x 2 Kolom)
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle(f'Evaluasi Model Terbaik: {champion_name}', fontsize=20, weight='bold')

# --- A. Confusion Matrix (Top Left) ---
# Fokus: Melihat seberapa banyak karyawan keluar yang berhasil ditebak (True Positive)
sns.heatmap(confusion_matrix(y_test, y_pred_test), annot=True, fmt='d', cmap='Blues',
            ax=axes[0, 0], cbar=False, annot_kws={"size": 14})
axes[0, 0].set_title('Confusion Matrix', fontsize=16)
axes[0, 0].set_xlabel('Predicted Label (0=Stay, 1=Leave)', fontsize=12)
axes[0, 0].set_ylabel('Actual Label (0=Stay, 1=Leave)', fontsize=12)
# axes[0, 0].text(0.5, 0.5, 'True Negative', ha='center', va='center', color='white', transform=axes[0,0].transData)


# --- B. ROC Curve (Top Right) ---
# Fokus: Trade-off antara Sensitivity dan Specificity
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_ylim([0.0, 1.05])
axes[0, 1].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 1].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 1].set_title('Receiver Operating Characteristic (ROC)', fontsize=16)
axes[0, 1].legend(loc="lower right")


# --- C. Learning Curve (Bottom Left) ---
# Fokus: Diagnosa Overfitting (Celah besar antara Train & Val) atau Underfitting (Skor rendah keduanya)
print("Sedang mengkalkulasi Learning Curve (mungkin memakan waktu sedikit)...")
train_sizes, train_scores, test_scores = learning_curve(
    champion_model, X_train, y_train, cv=5, scoring='roc_auc',
    n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

axes[1, 0].plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
axes[1, 0].plot(train_sizes, test_mean, 'o-', color="g", label="Cross-validation score")
axes[1, 0].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
axes[1, 0].fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
axes[1, 0].set_title('Learning Curve (Bias vs Variance Check)', fontsize=16)
axes[1, 0].set_xlabel('Training Examples', fontsize=12)
axes[1, 0].set_ylabel('ROC-AUC Score', fontsize=12)
axes[1, 0].legend(loc="best")
axes[1, 0].grid(True)


# --- D. Feature Importance (Bottom Right) ---
# Fokus: Top Drivers of Attrition (Insight Bisnis)
# Logika ekstraksi fitur tergantung tipe model

feature_names = X.columns
importances = None

# Mengambil estimator asli dari pipeline (step bernama 'model')
final_estimator = champion_model.named_steps['model']

try:
    if hasattr(final_estimator, 'feature_importances_'):
        # Untuk Random Forest, XGBoost, Decision Tree
        importances = final_estimator.feature_importances_
    elif hasattr(final_estimator, 'coef_'):
        # Untuk Logistic Regression (ambil nilai absolut koefisien)
        importances = np.abs(final_estimator.coef_[0])

    if importances is not None:
        # Buat DataFrame
        feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
        feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(10) # Top 10

        # Plot
        sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis', ax=axes[1, 1])
        axes[1, 1].set_title('Top 10 Drivers of Employee Attrition', fontsize=16)
        axes[1, 1].set_xlabel('Importance Score', fontsize=12)
        axes[1, 1].set_ylabel('')
    else:
        # Fallback jika model tidak punya feature importance (misal KNN)
        axes[1, 1].text(0.5, 0.5, 'Feature Importance tidak tersedia untuk model ini (e.g., KNN)',
                        ha='center', va='center', fontsize=14)
        axes[1, 1].set_title('Feature Importance', fontsize=16)

except Exception as e:
    axes[1, 1].text(0.5, 0.5, f'Error plotting importance: {str(e)}', ha='center', va='center')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# Fokus: Melihat seberapa banyak karyawan keluar yang berhasil ditebak (True Positive)
sns.heatmap(confusion_matrix(y_test, y_pred_test), annot=True, fmt='d', cmap='Blues',
            ax=ax, cbar=False, annot_kws={"size": 14})
ax.set_title('Confusion Matrix', fontsize=16)
ax.set_xlabel('Predicted Label (0=Stay, 1=Leave)', fontsize=12)
ax.set_ylabel('Actual Label (0=Stay, 1=Leave)', fontsize=12)
plt.tight_layout()
plt.show()

## ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# Fokus: Trade-off antara Sensitivity dan Specificity
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Receiver Operating Characteristic (ROC)', fontsize=16)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Learning Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
# Fokus: Diagnosa Overfitting (Celah besar antara Train & Val) atau Underfitting (Skor rendah keduanya)

train_sizes, train_scores, test_scores = learning_curve(
    champion_model, X_train, y_train, cv=5, scoring='roc_auc',
    n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

ax.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
ax.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-validation score")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")
ax.set_title('Learning Curve (Bias vs Variance Check)', fontsize=16)
ax.set_xlabel('Training Examples', fontsize=12)
ax.set_ylabel('ROC-AUC Score', fontsize=12)
ax.legend(loc="best")
ax.grid(True)
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
# Fokus: Top Drivers of Attrition (Insight Bisnis)
# Logika ekstraksi fitur tergantung tipe model

feature_names = X.columns
importances = None

# Mengambil estimator asli dari pipeline (step bernama 'model')
final_estimator = champion_model.named_steps['model']

try:
    if hasattr(final_estimator, 'feature_importances_'):
        # Untuk Random Forest, XGBoost, Decision Tree
        importances = final_estimator.feature_importances_
    elif hasattr(final_estimator, 'coef_'):
        # Untuk Logistic Regression (ambil nilai absolut koefisien)
        importances = np.abs(final_estimator.coef_[0])

    if importances is not None:
        # Buat DataFrame
        feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
        feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(10) # Top 10

        # Plot
        sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis', ax=ax)
        ax.set_title('Top 10 Drivers of Employee Attrition', fontsize=16)
        ax.set_xlabel('Importance Score', fontsize=12)
        ax.set_ylabel('')
    else:
        # Fallback jika model tidak punya feature importance (misal KNN)
        ax.text(0.5, 0.5, 'Feature Importance tidak tersedia untuk model ini (e.g., KNN)',
                        ha='center', va='center', fontsize=14)
        ax.set_title('Feature Importance', fontsize=16)

except Exception as e:
    ax.text(0.5, 0.5, f'Error plotting importance: {str(e)}', ha='center', va='center')

plt.tight_layout()
plt.show()

In [ ]:
import joblib
import json

# 1. Simpan Pipeline Model Terbaik (Termasuk Scaler & Model)
# Menggunakan .pkl adalah standar untuk scikit-learn/XGBoost
joblib.dump(champion_model, 'attrition_model_final.pkl')

# 2. Simpan Daftar Fitur (Feature Names)
# Penting agar Dashboard tahu urutan kolom yang benar saat input data
feature_cols = X_train.columns.tolist()
with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)

print("Aset berhasil disimpan! Siap untuk di-push ke GitHub.")